# 19 · Ordered slice aggregator
Reads notebook 18's per-slice outputs (24 slices × 512 features, plus 24 × 12 logits per scan) and learns **how to combine the slices, in order**, instead of a plain average.

Two aggregators, both built as **"baseline + correction"**: they start out equal to your current mean-of-slices-6-18 prediction and only learn changes that help.
- **A · LogitPool**: tiny. Learns, for each finding, which slice positions to trust (about 300 numbers).
- **B · FeatAgg**: small transformer over the 24 slice features, with slice position, so it can learn cross-slice patterns.

Each is trained with 3 seeds and the seeds are averaged (on 58 val knees a single run is too noisy to trust).
**Settings:** GPU optional (CPU works, just slower) · Inputs: notebook `18` output.

In [ ]:
# Cell 1: setup
import glob, numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as Fn
from sklearn.metrics import roc_auc_score

LABELS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA',
          'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
PLANES = ['sagittal', 'coronal', 'axial']
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
LO, HI = 6, 19          # slices 6..18 = the baseline's mean window
SEEDS = [0, 1, 2, 3, 4]
EPOCHS = 15

def find(name):
    f = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    assert len(f) == 1, (name, f)
    return f[0]
print(DEVICE)

In [ ]:
# Cell 2: load notebook 18's outputs for all planes into one table
F, L, ids = [], [], []
for p in PLANES:
    F.append(np.load(find(f'feats_{p}.npy')))                 # (N, 24, 512) float16
    L.append(np.load(find(f'logits_{p}.npy')).astype(np.float32))   # (N, 24, 12)
    ids.append(pd.read_csv(find(f'ids_{p}.csv')).assign(plane=p))
F, L = np.concatenate(F), np.concatenate(L)
ids = pd.concat(ids).reset_index(drop=True)
assert len(F) == len(L) == len(ids)

Y = ids[LABELS].values.astype(np.float32)        # labels the features' model (14) was trained on; val rows = real labels
tr = np.where(ids['split'] == 'train')[0]
va = np.where(ids['split'] == 'val')[0]
print('scans:', len(ids), '| train', len(tr), '| val', len(va), '| feats', F.shape)

In [ ]:
# Cell 3: knee-level scoring (average the planes per knee, exactly like the submission)
truth = (ids.iloc[va].drop_duplicates('StudyInstanceUID').set_index('StudyInstanceUID')[LABELS] >= 0.5)

def knee_auc(prob_va):                       # prob_va: (len(va), 12) probabilities for the val scans
    df = pd.DataFrame(prob_va, columns=LABELS).assign(StudyInstanceUID=ids['StudyInstanceUID'].values[va])
    k = df.groupby('StudyInstanceUID')[LABELS].mean().loc[truth.index]
    per = pd.Series({l: roc_auc_score(truth[l], k[l]) for l in LABELS})
    return per.mean(), per

sig = lambda x: 1 / (1 + np.exp(-x))
base_auc, base_per = knee_auc(sig(L[va, LO:HI]).mean(1))
print(f'baseline (mean of slices 6-18): {base_auc:.4f}   <- should be ~0.789, the number to beat')

In [ ]:
# Cell 4: the two aggregators. Both = baseline logit + a learned correction that starts at 0.
class LogitPool(nn.Module):
    # per finding: a weight for every slice position (start: equal weight on 6-18, i.e. the baseline)
    def __init__(self):
        super().__init__()
        init = torch.full((24, 12), -4.0); init[LO:HI] = 0.0           # softmax -> ~uniform over 6-18
        self.w = nn.Parameter(init)
        self.scale = nn.Parameter(torch.ones(12)); self.bias = nn.Parameter(torch.zeros(12))
    def forward(self, feats, logits):
        att = torch.softmax(self.w, dim=0)                             # (24, 12), sums to 1 over slices
        pooled = (logits * att).sum(1)                                 # (B, 12)
        return pooled * self.scale + self.bias

class FeatAgg(nn.Module):
    # small transformer over the 24 slice features, with slice position; output added to the baseline
    def __init__(self, d=128, drop=0.3):
        super().__init__()
        self.inp = nn.Sequential(nn.LayerNorm(512), nn.Dropout(drop), nn.Linear(512, d), nn.GELU())
        self.pos = nn.Parameter(torch.zeros(1, 24, d))                 # learned "which slice am I"
        layer = nn.TransformerEncoderLayer(d, nhead=4, dim_feedforward=2 * d, dropout=drop, batch_first=True)
        self.enc = nn.TransformerEncoder(layer, num_layers=1)
        self.query = nn.Linear(d, 12)                                  # attention pooling: 12 slice-weightings, one per finding
        self.out = nn.Linear(d, 12)
        nn.init.zeros_(self.out.weight); nn.init.zeros_(self.out.bias) # correction starts at exactly 0
    def forward(self, feats, logits):
        h = self.enc(self.inp(feats) + self.pos)                       # (B, 24, d)
        a = torch.softmax(self.query(h), dim=1)                        # (B, 24, 12)
        pooled = torch.einsum('bsf,bsd->bfd', a, h)                    # (B, 12, d)
        corr = (self.out.weight.unsqueeze(0) * pooled).sum(-1) + self.out.bias   # (B, 12)
        return logits[:, LO:HI].mean(1) + corr

In [ ]:
# Cell 5: training loop (keeps the epoch with the best knee-level val AUC)
def batches(idx, bs, shuffle):
    idx = np.random.permutation(idx) if shuffle else idx
    for i in range(0, len(idx), bs):
        j = idx[i:i + bs]
        yield (torch.from_numpy(F[j].astype(np.float32)).to(DEVICE),
               torch.from_numpy(L[j]).to(DEVICE), torch.from_numpy(Y[j]).to(DEVICE))

@torch.no_grad()
def predict(model, idx):
    model.eval()
    return np.concatenate([torch.sigmoid(model(f, l)).cpu().numpy() for f, l, _ in batches(idx, 512, False)])

def train(make, seed, lr, epochs=EPOCHS):
    torch.manual_seed(seed); np.random.seed(seed)
    model = make().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    best, best_state, hist = -1, None, []
    for ep in range(epochs):
        model.train(); tot = 0
        for f, l, y in batches(tr, 128, True):
            f = f + 0.1 * torch.randn_like(f)                          # feature noise: fights the "too-clean train features" problem
            loss = Fn.binary_cross_entropy_with_logits(model(f, l), y)
            opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item() * len(y)
        auc, _ = knee_auc(predict(model, va))
        hist.append((ep, tot / len(tr), auc))
        if auc > best: best, best_state = auc, {k: v.detach().clone() for k, v in model.state_dict().items()}
    return model, best, hist

In [ ]:
# Cell 6: run both aggregators with 3 seeds each
results, preds, models = {}, {}, {}
for name, make, lr in [('B_featagg', FeatAgg, 3e-4)]:
    ps, ms = [], []
    for s in SEEDS:
        m, best, hist = train(make, s, lr)
        print(f'{name} seed {s}: best epoch {max(hist, key=lambda h: h[2])[0]:2d}  val {best:.4f}  '
              f'| last epoch val {hist[-1][2]:.4f}  train loss {hist[-1][1]:.4f}')
        ps.append(predict(m, va)); ms.append(m)
    preds[name] = np.mean(ps, 0); models[name] = ms
    results[name] = knee_auc(preds[name])

print(f'\nbaseline mean 6-18      : {base_auc:.4f}')
for name, (auc, _) in results.items():
    print(f'{name:12s} (3 seeds avg): {auc:.4f}   change vs baseline: {auc - base_auc:+.4f}')

In [ ]:
# Cell 7: per-finding comparison + save the aggregator weights for the submission
table = pd.DataFrame({'baseline': base_per, **{n: r[1] for n, r in results.items()}})
print(table.round(3)); print('\nmacro:', table.mean().round(4).to_dict())

for name, ms in models.items():
    for s, m in zip(SEEDS, ms):
        torch.save(m.state_dict(), f'/kaggle/working/agg_{name}_seed{s}.pt')
print('saved:', sorted(glob.glob('/kaggle/working/agg_*.pt')))